In [1]:
import numpy as np


class NeuralNetwork:

    def __init__(self, input_dim, hidden_dim, output_dim, learning_rate=0.1):
        self.lr = learning_rate

        # Initialize weights and biases (Xavier/Glorot Initialization)
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(
            2.0 / input_dim
        )
        self.b1 = np.zeros((1, hidden_dim))

        self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(
            2.0 / hidden_dim
        )
        self.b2 = np.zeros((1, output_dim))

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def _sigmoid_derivative(self, a):
        # Derivative in terms of sigmoid output 'a'
        return a * (1 - a)

    def forward(self, X):
        """Forward pass: computes predictions given input X."""
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self._sigmoid(self.z1)

        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = self._sigmoid(self.z2)

        return self.a2

    def backward(self, X, y):
        """Backward pass: computes gradients and updates weights using gradient descent."""
        m = X.shape[0]  # Number of samples

        # Output layer error (Binary Cross-Entropy gradient w.r.t z2 simplifies to a2 - y)
        dz2 = self.a2 - y
        dW2 = (1 / m) * np.dot(self.a1.T, dz2)
        db2 = (1 / m) * np.sum(dz2, axis=0, keepdims=True)

        # Hidden layer error (Chain Rule)
        da1 = np.dot(dz2, self.W2.T)
        dz1 = da1 * self._sigmoid_derivative(self.a1)
        dW1 = (1 / m) * np.dot(X.T, dz1)
        db1 = (1 / m) * np.sum(dz1, axis=0, keepdims=True)

        # Gradient Descent Updates
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def train(self, X, y, epochs=10000):
        for epoch in range(epochs):
            # 1. Forward Pass
            y_pred = self.forward(X)

            # 2. Compute Loss (Binary Cross Entropy)
            loss = -np.mean(
                y * np.log(y_pred + 1e-15) + (1 - y) * np.log(1 - y_pred + 1e-15)
            )

            # 3. Backward Pass
            self.backward(X, y)

            if epoch % 2000 == 0:
                print(f"Epoch {epoch:5d} | Loss: {loss:.6f}")


# ==========================================
# Testing the ANN on XOR Gate
# ==========================================
if __name__ == "__main__":
    # Inputs (X) and expected targets (y) for XOR
    X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
    y = np.array([[0], [1], [1], [0]])

    # Initialize network: 2 input nodes, 4 hidden nodes, 1 output node
    model = NeuralNetwork(
        input_dim=2, hidden_dim=4, output_dim=1, learning_rate=0.5
    )

    print("--- Training NumPy Neural Network ---")
    model.train(X, y, epochs=10000)

    # Output final predictions
    predictions = model.forward(X)
    print("\n--- Final Predictions ---")
    for inputs, pred, target in zip(X, predictions, y):
        print(
            f"Input: {inputs} | Predicted: {pred[0]:.4f} (Class: {round(pred[0])}) | Target: {target[0]}"
        )

--- Training NumPy Neural Network ---
Epoch     0 | Loss: 0.720157
Epoch  2000 | Loss: 0.013030
Epoch  4000 | Loss: 0.004823
Epoch  6000 | Loss: 0.002931
Epoch  8000 | Loss: 0.002100

--- Final Predictions ---
Input: [0 0] | Predicted: 0.0006 (Class: 0) | Target: 0
Input: [0 1] | Predicted: 0.9977 (Class: 1) | Target: 1
Input: [1 0] | Predicted: 0.9987 (Class: 1) | Target: 1
Input: [1 1] | Predicted: 0.0024 (Class: 0) | Target: 0
